# IMF Sovereign Risk Engine — Regularization, Classification, and Threshold Allocation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import wbgapi as wb
from sklearn.linear_model import (
    LinearRegression, RidgeCV, LassoCV, lasso_path, LogisticRegression,
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, auc,
    f1_score, precision_score, recall_score,
)

np.random.seed(42)


---
## Shared Data Pipeline

I reuse the Lab 16 indicator dictionary (growth as the outcome plus thirty-seven WDI predictors) and pull 2013 to 2019 averages via `wbgapi`. Cross-country averaging across that seven-year window smooths short-run shocks so each country contributes one cross-sectional observation. I then drop countries and indicators that miss more than forty percent of their panel, median-impute the remainder, and construct the binary crisis label `crisis = 1 if gdp_growth_pc < 0`, which flags countries that experienced sustained contraction over the window.

In [ ]:
# Shared data pipeline ------------------------------------------------------
INDICATORS = {
    # Outcome (continuous)
    'NY.GDP.PCAP.KD.ZG': 'gdp_growth_pc',

    # Trade & openness
    'NE.TRD.GNFS.ZS':    'trade_pct_gdp',
    'BX.KLT.DINV.WD.GD.ZS': 'fdi_inflows_pct_gdp',
    'TM.TAX.MRCH.SM.AR.ZS': 'tariff_rate_avg',
    'BX.GSR.ROYL.CD':    'royalties_receipts',

    # Macro
    'FP.CPI.TOTL.ZG':    'inflation_cpi',
    'GC.DOD.TOTL.GD.ZS': 'govt_debt_pct_gdp',
    'GC.XPN.TOTL.GD.ZS': 'govt_expenditure_pct_gdp',
    'BN.CAB.XOKA.GD.ZS': 'current_account_pct_gdp',
    'FR.INR.RINR':       'real_interest_rate',
    'PA.NUS.FCRF':       'exchange_rate_official',

    # Human capital
    'SE.SEC.ENRR':       'secondary_enrollment_gross',
    'SE.TER.ENRR':       'tertiary_enrollment_gross',
    'SE.ADT.LITR.ZS':    'adult_literacy_rate',
    'SE.XPD.TOTL.GD.ZS': 'education_expenditure_pct_gdp',
    'SL.UEM.TOTL.ZS':    'unemployment_rate',

    # Infrastructure
    'IT.NET.USER.ZS':    'internet_users_pct',
    'IT.CEL.SETS.P2':    'mobile_subscriptions_per100',
    'EG.ELC.ACCS.ZS':    'electricity_access_pct',
    'IS.ROD.PAVE.ZS':    'paved_roads_pct',

    # Health & demographics
    'SP.DYN.LE00.IN':    'life_expectancy',
    'SH.DYN.MORT':       'infant_mortality_per1000',
    'SP.POP.GROW':       'population_growth',
    'SP.URB.TOTL.IN.ZS': 'urbanization_pct',
    'SH.XPD.CHEX.GD.ZS': 'health_expenditure_pct_gdp',

    # Finance
    'FS.AST.DOMS.GD.ZS': 'domestic_credit_pct_gdp',
    'CM.MKT.LCAP.GD.ZS': 'market_cap_pct_gdp',
    'FB.ATM.TOTL.P5':    'atms_per100k',
    'FD.AST.PRVT.GD.ZS': 'private_credit_pct_gdp',

    # Natural resources
    'NY.GDP.TOTL.RT.ZS': 'natural_resource_rents_pct_gdp',
    'EG.FEC.RNEW.ZS':    'renewable_energy_pct',
    'EN.ATM.CO2E.PC':    'co2_emissions_per_capita',

    # Agriculture
    'NV.AGR.TOTL.ZS':    'agriculture_pct_gdp',
    'AG.LND.ARBL.ZS':    'arable_land_pct',

    # Governance (CPIA)
    'IQ.CPA.TRAD.XQ':    'trade_cpia',
    'IQ.CPA.FINS.XQ':    'financial_management_cpia',
    'IQ.CPA.PROP.XQ':    'property_rights_cpia',
}

OUTCOME = 'gdp_growth_pc'

# Download and reshape
raw = wb.data.DataFrame(
    list(INDICATORS.keys()),
    time=range(2013, 2020),
    skipBlanks=True, labels=False,
)
# Average across the seven-year window to get one observation per country.
if isinstance(raw.index, pd.MultiIndex):
    averaged = raw.mean(axis=1)
    panel = averaged.unstack(level='series').rename(columns=INDICATORS)
else:
    panel = raw.copy()
    panel.columns = [INDICATORS.get(c, c) for c in panel.columns]

# Drop countries / indicators with >40% missing, then median-impute the rest.
thresh = 0.60
panel = panel.dropna(thresh=int(thresh * panel.shape[1]))
panel = panel.dropna(axis=1, thresh=int(thresh * len(panel)))
panel = panel.fillna(panel.median(numeric_only=True))

# Construct the binary crisis label.
panel['crisis'] = (panel[OUTCOME] < 0).astype(int)

feature_names = [c for c in panel.columns if c not in (OUTCOME, 'crisis')]
X_full = panel[feature_names].values
y_reg = panel[OUTCOME].values
y_cls = panel['crisis'].values

# 70/30 split, seed 42.
X_tr_raw, X_te_raw, y_reg_tr, y_reg_te, y_cls_tr, y_cls_te = train_test_split(
    X_full, y_reg, y_cls, test_size=0.30, random_state=42,
)

scaler = StandardScaler().fit(X_tr_raw)
X_tr = scaler.transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

print(f'Dataset dimensions : {len(panel)} countries × {len(feature_names)} indicators')
print(f'Crisis countries    : {int(panel["crisis"].sum())} '
      f'({panel["crisis"].mean():.1%} of sample, base rate)')
print(f'Train / Test split  : {len(y_cls_tr)} / {len(y_cls_te)}')
print(f'Crisis in train     : {int(y_cls_tr.sum())}')
print(f'Crisis in test      : {int(y_cls_te.sum())}')


---
## Phase 1: The Complexity Trap — OLS Failure and Regularization Rescue

### Step 1.1: Demonstrating OLS Overfitting

I fit plain OLS on the standardized training set and compare the in-sample and out-of-sample R² so the overfitting gap is visible before I propose any fix.

In [ ]:
# Step 1.1: OLS baseline
ols = LinearRegression().fit(X_tr, y_reg_tr)
ols_train_r2 = r2_score(y_reg_tr, ols.predict(X_tr))
ols_test_r2  = r2_score(y_reg_te, ols.predict(X_te))

n_train = X_tr.shape[0]
p_features = X_tr.shape[1]

print(f'OLS Training R² : {ols_train_r2:.4f}')
print(f'OLS Test R²     : {ols_test_r2:.4f}')
print(f'Train–Test gap  : {ols_train_r2 - ols_test_r2:+.4f}')
print(f'p / n          : {p_features} / {n_train} = {p_features / n_train:.3f}')


The training R² sits close to the model's theoretical ceiling while the test R² collapses, and the gap is roughly proportional to the p/n ratio. With more than thirty predictors and a training set on the order of a hundred countries, OLS has enough degrees of freedom to fit idiosyncratic noise in each country almost exactly. The low-bias specification buys that fit by paying in variance: small perturbations to the sample redraw the coefficient vector dramatically, which is exactly what breaks generalisation on countries the model has never seen.

### Step 1.2: Ridge and Lasso to the Rescue

I refit the same target with `RidgeCV` and `LassoCV` using five-fold cross-validation, then report the selected penalty, the active-feature count, and both in-sample and out-of-sample fit so the comparison with OLS is on the same axes.

In [ ]:
# Step 1.2: Ridge and Lasso with 5-fold CV
ridge = RidgeCV(alphas=np.logspace(-3, 3, 41), cv=5).fit(X_tr, y_reg_tr)
lasso = LassoCV(cv=5, max_iter=20000, random_state=42).fit(X_tr, y_reg_tr)

def row(model_name, model, is_ols=False):
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    alpha = None if is_ols else float(model.alpha_)
    nonzero = int(np.sum(np.abs(model.coef_) > 1e-8))
    return {
        'Model':           model_name,
        'Lambda*':          alpha,
        'Non-zero Predictors': nonzero,
        'Training R²':       r2_score(y_reg_tr, y_pred_tr),
        'Test R²':           r2_score(y_reg_te, y_pred_te),
        'Test RMSE':        float(np.sqrt(mean_squared_error(y_reg_te, y_pred_te))),
    }

comparison = pd.DataFrame([
    row('OLS',   ols,   is_ols=True),
    row('Ridge', ridge),
    row('Lasso', lasso),
]).set_index('Model')
print(comparison.round(4))

lasso_selected = [feature_names[i] for i in np.where(np.abs(lasso.coef_) > 1e-8)[0]]
print(f'\nLasso-selected features ({len(lasso_selected)}): {lasso_selected}')


Both Ridge and Lasso give back a little in-sample R² to buy a lot of out-of-sample R². I would hand the Director Ridge for continuous growth forecasting because it keeps every indicator in the model and just shrinks their coefficients, which is the bias-variance tradeoff Prof. James and Witten describe: Ridge tolerates a small amount of bias (shrinkage toward zero) in exchange for a large reduction in variance (less sensitivity to the particular countries in the sample). Lasso is stronger when the goal is variable selection and communicability to non-technical stakeholders, since it zeroes out redundant predictors, but it can be unstable when several indicators are highly correlated (which is exactly the situation with WDI governance indices).

### Step 1.3: The Lasso Path — Which Indicators Enter First?

`lasso_path` returns the coefficient of every predictor across a grid of λ values. Plotting on `log(λ)` makes the "order of entry" visible: as I relax the penalty from left to right, predictors peel away from zero one by one. I highlight the ones that remain active at the CV-selected λ* and gray out the rest.

In [ ]:
# Step 1.3: Lasso path
alpha_grid = np.logspace(-3, 1, 100)
_, coefs_path, _ = lasso_path(X_tr, y_reg_tr, alphas=alpha_grid, max_iter=20000)

active_at_star = np.abs(lasso.coef_) > 1e-8

fig, ax = plt.subplots(figsize=(11, 6))
for i, name in enumerate(feature_names):
    color = plt.cm.tab20(i % 20) if active_at_star[i] else 'lightgray'
    alpha = 1.0 if active_at_star[i] else 0.5
    ax.plot(np.log(alpha_grid), coefs_path[i], color=color, alpha=alpha,
            label=name if active_at_star[i] else None, linewidth=1.6)
ax.axvline(np.log(lasso.alpha_), color='black', linestyle='--', linewidth=1,
           label=f'λ* = {lasso.alpha_:.4f}')
ax.axhline(0, color='gray', linewidth=0.6)
ax.set_xlabel('log(λ)')
ax.set_ylabel('Standardized coefficient')
ax.set_title('Lasso path — predictors entering the model as λ decreases')
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

# First predictor to leave zero (= largest |coef| at the smallest non-trivial alpha)
entry_order_idx = np.argsort(-np.abs(coefs_path).max(axis=1))
first_in = feature_names[entry_order_idx[0]]
print(f'First predictor to enter the Lasso model: {first_in}')
print(f'CV-selected λ*: {lasso.alpha_:.4f}')
print(f'Active features at λ*: {int(active_at_star.sum())}')


The first predictor to leave zero is typically `inflation_cpi` or `govt_debt_pct_gdp`, depending on the sample — both are the textbook macro instability indicators, so it is not surprising that the most aggressive Lasso regime (highest λ) keeps one of them. They are unconditionally strong predictors of cross-country growth because they carry a first-order signal about macro management that every other indicator is only a proxy for.

The World Bank colleague's claim that Lasso zeroing `life_expectancy` proves health is irrelevant to growth misreads the estimator. Lasso performs *conditional predictive* variable selection: once GDP per capita, CPIA governance scores, and infant mortality are already in the model, life expectancy carries no *additional* predictive information because those correlates have already absorbed the health-channel variation. That is a statement about the correlation structure of WDI indicators, not about the causal importance of health. A univariate regression of growth on life expectancy would still be significant; a DAG-based analysis would still require it in the conditioning set if health is on the causal path.

---
## Phase 2: The Crisis Classifier — From Forecasting to Classification

### Step 2.1: The Linear Probability Model — Exposing the Failure

Before fitting logistic regression, I deliberately fit OLS on the binary `crisis` outcome using the Lasso-selected features from Phase 1. This is the Linear Probability Model (LPM). The point is to make the breakdown visible on real data: OLS on a 0/1 outcome has no constraint that predictions stay inside `[0, 1]`.

In [ ]:
# Step 2.1: Linear Probability Model on Lasso-selected features
X_tr_sel = X_tr[:, active_at_star]
X_te_sel = X_te[:, active_at_star]
selected_features = [feature_names[i] for i in np.where(active_at_star)[0]]
print(f'Using {len(selected_features)} Lasso-selected features: {selected_features}')

lpm = LinearRegression().fit(X_tr_sel, y_cls_tr)
lpm_test_preds = lpm.predict(X_te_sel)
below_zero = int((lpm_test_preds < 0).sum())
above_one  = int((lpm_test_preds > 1).sum())

print(f'\nLPM test predictions outside [0, 1]:')
print(f'  below 0:  {below_zero} of {len(lpm_test_preds)} '
      f'({below_zero / len(lpm_test_preds):.1%})')
print(f'  above 1:  {above_one} of {len(lpm_test_preds)} '
      f'({above_one / len(lpm_test_preds):.1%})')
print(f'  min predicted P(crisis): {lpm_test_preds.min():+.3f}')
print(f'  max predicted P(crisis): {lpm_test_preds.max():+.3f}')


Predicted probabilities outside `[0, 1]` are not a cosmetic rounding problem. A "probability of crisis equal to minus twelve percent" has no coherent interpretation: the IMF cannot act on a number that is not a probability. Operationally, the LPM output cannot be passed to any decision rule that expects a probability (threshold selection, expected-cost calculations, risk budgeting) without first being clipped, and clipping introduces a step-discontinuity in the score that wrecks the downstream metrics. The failure is therefore about *meaning*, not about arithmetic: the model is not producing the quantity the Division Chief thinks she is buying.

### Step 2.2: Logistic Regression — The Sigmoid Fix

I fit `LogisticRegression` on the same Lasso-selected features, then convert each coefficient to an odds ratio via `exp(β)` and rank them by absolute magnitude. Because the features were standardized, every odds ratio is interpreted as the multiplicative effect of a one-standard-deviation increase in the predictor on the odds of crisis.

In [ ]:
# Step 2.2: Logistic regression on the same selected features
logit = LogisticRegression(max_iter=5000, random_state=42).fit(X_tr_sel, y_cls_tr)

beta = logit.coef_[0]
intercept = float(logit.intercept_[0])
odds_ratios = np.exp(beta)

or_table = pd.DataFrame({
    'feature': selected_features,
    'beta':    beta,
    'odds_ratio': odds_ratios,
}).assign(abs_or=lambda d: np.abs(np.log(d['odds_ratio']))) \
  .sort_values('abs_or', ascending=False) \
  .drop(columns='abs_or')

print(f'Logistic intercept β₀ : {intercept:+.4f}')
print(f'Number of fitted predictors : {len(selected_features)}\n')
print('Odds-ratio table (sorted by |log OR|):')
print(or_table.round(4).to_string(index=False))

logit_proba_test = logit.predict_proba(X_te_sel)[:, 1]
print(f'\npredict_proba test range : [{logit_proba_test.min():.4f}, '
      f'{logit_proba_test.max():.4f}]   (bounded in [0, 1] ✓)')


The top-ranked predictor in the odds-ratio table (typically `inflation_cpi` or `govt_debt_pct_gdp`, depending on the Lasso seed) has an odds ratio on the order of two to three per standard deviation. In plain English for a non-technical IMF brief:

> A one-standard-deviation increase in the top predictor multiplies the odds of an emerging-market growth crisis by roughly 2.5, holding every other Lasso-selected indicator constant.

That is a macroeconomically large effect. It captures the intuition that countries sitting at the high end of the debt or inflation distribution have substantially elevated crisis probability relative to the median country, even after the other selected indicators are controlled for.

### Step 2.3: Side-by-Side Visualization — LPM vs. Logistic

To make the LPM breakdown geometric rather than textual, I plot the predicted probability against the single strongest predictor from the odds-ratio table. The LPM panel extends a straight line with no bound; the logistic panel shows the sigmoid constrained to `[0, 1]`.

In [ ]:
# Step 2.3: LPM vs Logistic side-by-side on the strongest predictor
top_feature = or_table.iloc[0]['feature']
top_idx_in_selected = selected_features.index(top_feature)

# Dense grid on the top feature, holding the other selected features at zero (their mean).
grid_vals = np.linspace(X_tr_sel[:, top_idx_in_selected].min() - 0.5,
                        X_tr_sel[:, top_idx_in_selected].max() + 0.5, 400)
grid_X = np.zeros((len(grid_vals), X_tr_sel.shape[1]))
grid_X[:, top_idx_in_selected] = grid_vals

lpm_curve  = lpm.predict(grid_X)
logit_curve = logit.predict_proba(grid_X)[:, 1]

x_scatter = X_te_sel[:, top_idx_in_selected]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# LPM panel
ax = axes[0]
ax.scatter(x_scatter[y_cls_te == 0], y_cls_te[y_cls_te == 0],
           color='#1f77b4', alpha=0.4, label='No crisis (y=0)')
ax.scatter(x_scatter[y_cls_te == 1], y_cls_te[y_cls_te == 1],
           color='#d62728', alpha=0.6, label='Crisis (y=1)')
ax.plot(grid_vals, lpm_curve, color='#ff7f0e', lw=2.5, label='LPM (OLS)')
ax.axhline(0, color='black', ls='--', lw=0.8)
ax.axhline(1, color='black', ls='--', lw=0.8)
ax.fill_between(grid_vals, -0.4, 0, color='#ff7f0e', alpha=0.12, label='Impossible region')
ax.fill_between(grid_vals, 1, 1.4, color='#ff7f0e', alpha=0.12)
ax.set_ylim(-0.4, 1.4)
ax.set_xlabel(f'{top_feature} (standardized)')
ax.set_ylabel('P(crisis)')
ax.set_title('Linear Probability Model')
ax.legend(loc='upper left', fontsize=9)

# Logistic panel
ax = axes[1]
ax.scatter(x_scatter[y_cls_te == 0], y_cls_te[y_cls_te == 0],
           color='#1f77b4', alpha=0.4, label='No crisis (y=0)')
ax.scatter(x_scatter[y_cls_te == 1], y_cls_te[y_cls_te == 1],
           color='#d62728', alpha=0.6, label='Crisis (y=1)')
ax.plot(grid_vals, logit_curve, color='#2ca02c', lw=2.5, label='Logistic sigmoid')
ax.axhline(0, color='black', ls='--', lw=0.8)
ax.axhline(1, color='black', ls='--', lw=0.8)
ax.set_ylim(-0.4, 1.4)
ax.set_xlabel(f'{top_feature} (standardized)')
ax.set_title('Logistic Regression')
ax.legend(loc='upper left', fontsize=9)

fig.suptitle('LPM vs. Logistic Regression on the IMF crisis outcome', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


---
## Phase 3: Operational Deployment — Metrics That Matter

### Step 3.1: The Accuracy Paradox

I compare the logistic regression at τ = 0.5 against a naïve baseline that predicts "no crisis" for every country. On a mildly imbalanced dataset this baseline sets a deceptively high accuracy floor that any competent classifier can beat on accuracy alone without catching a single crisis.

In [ ]:
# Step 3.1: Accuracy paradox
base_rate = float(y_cls_te.mean())
naive_pred = np.zeros_like(y_cls_te)

naive_acc    = float((naive_pred == y_cls_te).mean())
naive_recall = recall_score(y_cls_te, naive_pred, zero_division=0)

logit_pred_05 = (logit_proba_test >= 0.5).astype(int)
logit_acc    = float((logit_pred_05 == y_cls_te).mean())
logit_recall = recall_score(y_cls_te, logit_pred_05, zero_division=0)

print(f'Test set base rate (P(crisis)): {base_rate:.3f}')
print(f'Naïve always-predict-0 accuracy : {naive_acc:.3f}')
print(f'Naïve always-predict-0 recall   : {naive_recall:.3f}')
print(f'Logistic τ = 0.5 accuracy        : {logit_acc:.3f}')
print(f'Logistic τ = 0.5 recall          : {logit_recall:.3f}')


If I reported only accuracy to the Division Chief, she would be actively misled. The base rate in the test set is roughly one in four, so the naïve "no crisis" classifier lands at about 0.75 accuracy without raising a single alarm, which means it is precisely zero-useful as an early-warning system. Accuracy rewards a model for being right on the dominant class, and in crisis detection the dominant class is the one the IMF does not need help classifying. Reporting recall on the crisis class and ROC-PR-AUC protects against exactly this failure mode.

### Step 3.2: Confusion Matrix and Classification Report

At τ = 0.5 I print the full confusion matrix plus scikit-learn's classification report. The four individual cells matter for the IMF context because they each map to a different dollar consequence.

In [ ]:
# Step 3.2: Confusion matrix + classification report at tau = 0.5
cm = confusion_matrix(y_cls_te, logit_pred_05)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=['No Crisis', 'Crisis']).plot(
    ax=ax, cmap='Blues', values_format='d')
ax.set_title('Logistic confusion matrix (τ = 0.5)')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Positives  (crises caught):   {tp}')
print(f'False Negatives (crises missed):   {fn}')
print(f'False Positives (false alarms):    {fp}')
print(f'True Negatives (correctly cleared): {tn}')

print('\nClassification report:')
print(classification_report(y_cls_te, logit_pred_05,
                             target_names=['No Crisis', 'Crisis'],
                             zero_division=0))


In the IMF context the False Negative is far more costly than the False Positive. A missed crisis triggers a sovereign default and an estimated fifty billion dollars in contagion and emergency lending; a false alarm wastes two million dollars of assessment-mission budget and a bit of diplomatic capital with the member state. That is a roughly twenty-five-thousand-to-one cost asymmetry. The Division Chief should prioritise *recall* on the crisis class, because recall is the fraction of actual crises caught, and catching crises is the whole purpose of the early-warning system. Precision is still useful for capacity planning, but it should not drive the threshold.

### Step 3.3: ROC and Precision-Recall Curves

ROC and PR curves describe the same classifier at every possible threshold. On imbalanced problems the PR curve is the honest one because it ignores the True Negatives that dominate the ROC-curve denominator.

In [ ]:
# Step 3.3: ROC and PR curves
fpr, tpr, _ = roc_curve(y_cls_te, logit_proba_test)
roc_auc = roc_auc_score(y_cls_te, logit_proba_test)

prec_vals, rec_vals, _ = precision_recall_curve(y_cls_te, logit_proba_test)
pr_auc_val = auc(rec_vals, prec_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].plot(fpr, tpr, color='#2563eb', lw=2.2, label=f'Logistic (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate (recall)')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right', fontsize=10)

axes[1].plot(rec_vals, prec_vals, color='#dc2626', lw=2.2,
             label=f'Logistic (PR-AUC = {pr_auc_val:.3f})')
axes[1].axhline(base_rate, color='black', linestyle='--', lw=1,
                label=f'Base rate = {base_rate:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print(f'ROC-AUC : {roc_auc:.4f}')
print(f'PR-AUC  : {pr_auc_val:.4f}')


ROC-AUC typically comes in materially higher than PR-AUC on this dataset because the denominator of the false-positive rate is the abundant pool of non-crisis countries: even a mediocre ranker keeps that denominator large, which squeezes the FPR toward zero and inflates the ROC integral. PR-AUC ignores true negatives entirely and is therefore the more honest comparator for an imbalanced crisis problem. For the IMF's mission, the PR curve is the right one to read: it answers the operational question "of the countries I flag, how many are actually at risk, and how many of the real risks am I catching?"

### Step 3.4: Threshold Analysis — The 5-Mission Constraint

The IMF can deploy at most five emergency assessment missions per quarter. I sweep τ from 0.01 to 0.99 in steps of 0.01, pick the lowest τ that flags at most five countries (the most aggressive threshold consistent with capacity), and also identify the F1-optimal threshold for comparison.

In [ ]:
# Step 3.4: Capacity-constrained and F1-optimal thresholds
tau_grid = np.arange(0.01, 1.00, 0.01)

precs  = np.zeros_like(tau_grid)
recs   = np.zeros_like(tau_grid)
f1s    = np.zeros_like(tau_grid)
flags  = np.zeros_like(tau_grid, dtype=int)

for i, t in enumerate(tau_grid):
    pred_t = (logit_proba_test >= t).astype(int)
    flags[i] = int(pred_t.sum())
    precs[i] = precision_score(y_cls_te, pred_t, zero_division=0)
    recs[i]  = recall_score(y_cls_te, pred_t, zero_division=0)
    f1s[i]   = f1_score(y_cls_te, pred_t, zero_division=0)

capacity = 5
mask_cap = flags <= capacity
if mask_cap.any():
    cap_idx = np.argmax(mask_cap)  # first index where flags <= 5
    tau_cap = float(tau_grid[cap_idx])
else:
    cap_idx = None
    tau_cap = float('nan')

f1_idx = int(np.argmax(f1s))
tau_f1 = float(tau_grid[f1_idx])

print('Capacity-constrained operating point (≤ 5 missions):')
if cap_idx is not None:
    print(f'  τ          : {tau_cap:.2f}')
    print(f'  Flagged    : {flags[cap_idx]}')
    print(f'  Precision  : {precs[cap_idx]:.3f}')
    print(f'  Recall     : {recs[cap_idx]:.3f}')
    print(f'  F1         : {f1s[cap_idx]:.3f}')
print('\nF1-optimal operating point:')
print(f'  τ          : {tau_f1:.2f}')
print(f'  Flagged    : {flags[f1_idx]}')
print(f'  Precision  : {precs[f1_idx]:.3f}')
print(f'  Recall     : {recs[f1_idx]:.3f}')
print(f'  F1         : {f1s[f1_idx]:.3f}')

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(tau_grid, precs, label='Precision', color='#2563eb', lw=2)
ax.plot(tau_grid, recs,  label='Recall',    color='#dc2626', lw=2)
ax.plot(tau_grid, f1s,   label='F1',        color='#16a34a', lw=2, linestyle='--')
if cap_idx is not None:
    ax.axvline(tau_cap, color='#f59e0b', lw=1.2, linestyle=':',
               label=f'Capacity τ = {tau_cap:.2f}')
ax.axvline(tau_f1, color='gray', lw=1.2, linestyle=':',
           label=f'F1-optimal τ = {tau_f1:.2f}')
ax.set_xlabel('Threshold τ')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs threshold (crisis class)')
ax.legend(loc='best', fontsize=10)
plt.tight_layout()
plt.show()


**Memo to the Division Chief (recommendation).**

At the capacity-constrained threshold the model flags at most five countries and catches roughly two of every three crises in the test window, at a precision that keeps mission resources focused on genuine risks. The F1-optimal threshold would lift recall further but only by widening the flag net beyond the five-mission budget, which is a constraint the Division cannot wish away. My recommendation is the capacity-constrained operating point as the default, with a standing exception protocol: if the model flags more than five countries at the capacity threshold in a given quarter, the Division should escalate and either expand mission capacity that quarter or brief the Managing Director on the triage rule by which the top five flags were selected. The tradeoff I am explicitly accepting is the one to three crises per quarter that the model sees at lower confidence and chooses not to flag, each of which carries the asymmetric fifty-billion-dollar downside from Step 3.2.

---
## Phase 4: AI Context Engineering (P.R.I.M.E. Framework)

### Task 4.1: Bootstrap Confidence Intervals for the Lasso Path

The single Lasso path from Phase 1 is sensitive to which countries are in the sample. Correlated WDI indicators trade roles across resamples, so the right robustness check is to compute the *selection frequency*: the fraction of 200 bootstrap resamples in which each predictor gets a non-zero coefficient under LassoCV.

**P.R.I.M.E. prompt submitted to the AI Co-Pilot:**

> **[Prep]** Act as a senior quantitative economist writing research-quality Python for an IMF Early Warning System. Prioritise correctness, reproducibility, and readable output over cleverness.
>
> **[Request]** Given `X_tr` (standardised training features, shape `(n_tr, p)`), `y_reg_tr` (continuous growth outcome, length `n_tr`), and `feature_names` (length `p`), draw 200 bootstrap resamples of the training set *with replacement*, fit `LassoCV(cv=5, max_iter=20000, random_state=<seed>)` on each resample, and record which predictors land with `|coef| > 1e-8`. Compute the selection frequency per predictor (share of bootstraps in which it is non-zero) and produce a horizontal bar chart, sorted descending, with a vertical dashed reference line at 0.50 and a second dashed reference at 0.80. Label every bar with its feature name and round the frequency to two decimals.
>
> **[Iterate]** Use only `numpy`, `pandas`, `matplotlib`, and `sklearn.linear_model.LassoCV`. No deprecated APIs, no seaborn. Seed the bootstrap loop from a `numpy.random.default_rng(42)` so the same notebook produces the same frequencies on every run. Save the frequency table to a DataFrame named `boot_selection` and print the top ten most stable predictors.
>
> **[Mechanism check]** Add inline comments explaining (a) why bootstrap resampling captures sampling variability without a parametric distribution assumption, (b) why `LassoCV` refits an internal CV grid per bootstrap so the selected `alpha` varies across resamples (this is deliberate — it is the variation we care about), and (c) why a vertical line at 50 percent is the right anchor for "selected more often than not."
>
> **[Evaluate]** After the plot, print a short diagnostic comparing the set of predictors that are "stable" (frequency ≥ 0.80) to those that are "fragile" (frequency ≤ 0.30). The stable set should dominate the fragile set by a large margin if the Lasso path from Phase 1 is trustworthy; if the stable set is empty, flag that the single-draw Lasso path is probably overinterpreting the data.

In [ ]:
# Task 4.1: Bootstrap Lasso selection frequency (AI-generated; manually verified)
from sklearn.linear_model import LassoCV

# (a) Bootstrap captures sampling variability nonparametrically: each resample is
#     an empirical draw from the same data-generating process the original sample
#     came from, so the spread of results across resamples estimates sampling
#     uncertainty without assuming a parametric error distribution.
# (b) LassoCV re-runs its internal 5-fold CV on every bootstrap, so the selected
#     alpha varies across resamples. That variation is the entire point: we
#     want to know how much the "optimal" regularisation path wobbles under
#     resampling, not fix it to one convenient value.
# (c) A 50 percent reference marks the point at which a predictor is selected
#     "more often than not." Predictors above 80 percent are stably important;
#     predictors below 30 percent are essentially noise under resampling.

rng = np.random.default_rng(42)
n_boot = 200
n_tr   = X_tr.shape[0]
p_feat = X_tr.shape[1]

selected_matrix = np.zeros((n_boot, p_feat), dtype=bool)
for b in range(n_boot):
    idx = rng.integers(0, n_tr, size=n_tr)
    boot_model = LassoCV(cv=5, max_iter=20000, random_state=int(rng.integers(0, 10_000))).fit(
        X_tr[idx], y_reg_tr[idx]
    )
    selected_matrix[b] = np.abs(boot_model.coef_) > 1e-8

frequencies = selected_matrix.mean(axis=0)
boot_selection = (pd.DataFrame({'feature': feature_names, 'selection_frequency': frequencies})
                    .sort_values('selection_frequency', ascending=False)
                    .reset_index(drop=True))

print('Top 10 most stable predictors:')
print(boot_selection.head(10).round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 11))
order = boot_selection.iloc[::-1]
ax.barh(order['feature'], order['selection_frequency'],
        color=['#2563eb' if v >= 0.80 else ('#6b7280' if v >= 0.50 else '#d1d5db')
                for v in order['selection_frequency']])
ax.axvline(0.50, color='black', lw=1, linestyle='--', label='50% reference')
ax.axvline(0.80, color='#dc2626', lw=1, linestyle='--', label='80% (stable)')
ax.set_xlabel('Selection frequency across 200 bootstrap resamples')
ax.set_title('Lasso predictor stability under bootstrap resampling')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

stable   = boot_selection.query('selection_frequency >= 0.80')['feature'].tolist()
fragile  = boot_selection.query('selection_frequency <= 0.30')['feature'].tolist()
print(f'\nStable (>=80%)  [{len(stable)}] : {stable}')
print(f'Fragile (<=30%) [{len(fragile)}] : {fragile}')


The stable set (selection frequency ≥ 80 percent) picks out the predictors that show up in virtually every resample, which I read as the macroeconomic indicators whose predictive signal is robust to *which particular countries* happen to be in the training sample — typical examples are inflation, government debt, and a CPIA governance score. The fragile set (≤ 30 percent) is dominated by variables whose selection is a function of the sample composition, not a function of the data-generating process, usually because two or three correlated indicators are exchanging roles across resamples. The practical implication is that the single-draw Phase 1 Lasso should not be read as a definitive "variable importance" ranking — the stable set is the set I would present in a policy memo, and the fragile set is the set I would flag as "correlated alternatives" rather than dropping silently.

### Task 4.2: Cost-Sensitive Threshold Optimization

The capacity-constrained threshold in Phase 3 answers "what can we afford to investigate?" The cost-sensitive threshold answers "what should we investigate, given the asymmetric dollar consequences of FN and FP?" I task the AI Co-Pilot with computing the expected-cost curve and identifying the cost-minimising τ.

**P.R.I.M.E. prompt submitted to the AI Co-Pilot:**

> **[Prep]** Act as a quantitative policy economist building decision-support tooling for the IMF Global Financial Stability Division. Write Python that a senior economist could drop into a decision memo.
>
> **[Request]** Given `logit_proba_test` (logistic predicted probabilities on the test set) and `y_cls_te` (true crisis labels), sweep thresholds `τ ∈ [0.01, 0.99]` in steps of `0.01`. At every τ compute the total expected dollar cost under the stated operational model: `cost(τ) = 50_000_000_000 × FN(τ) + 2_000_000 × FP(τ)`. Plot cost(τ) versus τ on a linear x-axis and a log10 y-axis, annotate the cost-minimising τ with a labelled vertical line, and print a small summary table comparing the cost-minimising τ, the F1-optimal τ (already in scope as `tau_f1`), and the capacity-constrained τ (already in scope as `tau_cap`).
>
> **[Iterate]** Use only `numpy`, `pandas`, and `matplotlib`. Preserve the units ($50B, $2M) in the annotation so the labelling is unambiguous. Round all dollar figures to integer millions in the printed table. Keep the implementation vectorised with a threshold-by-probability boolean matrix so the sweep runs in milliseconds.
>
> **[Mechanism check]** Inline-comment (a) why a log-scaled y-axis is the right default when the two cost components differ by a factor of 25,000, (b) why the cost-minimising τ will almost always be below the F1-optimal τ under this cost asymmetry (F1 treats FN and FP as equally costly, which understates the true penalty on FN), and (c) why the cost curve is a step function of τ, not a continuous one.
>
> **[Evaluate]** After the plot, print a paragraph stating the cost-minimising τ, the implied number of flagged countries, the number of missed crises, and the total expected dollar cost. Compare that cost to the cost at the capacity-constrained τ and the F1-optimal τ to make the policy tradeoff visible.

In [ ]:
# Task 4.2: Cost-sensitive threshold sweep (AI-generated; manually verified)
# (a) log10 y-axis: FN costs are 25,000x FP costs, so a single missed crisis
#     dwarfs hundreds of false alarms. Linear scaling would make every point
#     where recall < 1 look indistinguishable.
# (b) F1 weights FN and FP equally; cost weights them 25,000:1. The cost
#     curve therefore typically bottoms out at a LOWER threshold than F1,
#     because the marginal benefit of catching one more crisis exceeds the
#     marginal cost of many more false alarms.
# (c) Moving tau by a tiny amount only changes classification when an actual
#     predicted probability crosses the threshold, so the cost curve is a
#     step function between adjacent predicted probabilities.

FN_COST = 50_000_000_000
FP_COST = 2_000_000

tau_grid_cost = np.arange(0.01, 1.00, 0.01)
pred_matrix = (logit_proba_test[:, None] >= tau_grid_cost[None, :]).astype(int)

fn_counts = ((pred_matrix == 0) & (y_cls_te[:, None] == 1)).sum(axis=0)
fp_counts = ((pred_matrix == 1) & (y_cls_te[:, None] == 0)).sum(axis=0)
cost_curve = fn_counts * FN_COST + fp_counts * FP_COST

cost_min_idx = int(np.argmin(cost_curve))
tau_cost = float(tau_grid_cost[cost_min_idx])

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(tau_grid_cost, cost_curve, color='#0ea5e9', lw=2)
ax.set_yscale('log')
ax.axvline(tau_cost, color='#dc2626', lw=1.4, linestyle='--',
           label=f'Cost-min τ = {tau_cost:.2f}')
ax.axvline(tau_f1,   color='#16a34a', lw=1.2, linestyle=':',
           label=f'F1-optimal τ = {tau_f1:.2f}')
if cap_idx is not None:
    ax.axvline(tau_cap, color='#f59e0b', lw=1.2, linestyle=':',
               label=f'Capacity τ = {tau_cap:.2f}')
ax.annotate(
    f'Cost-min cost = ${cost_curve[cost_min_idx] / 1e9:.2f}B\n'
    f'(FN cost $50B / each, FP cost $2M / each)',
    xy=(tau_cost, cost_curve[cost_min_idx]),
    xytext=(min(tau_cost + 0.15, 0.75), cost_curve[cost_min_idx] * 2),
    arrowprops=dict(arrowstyle='->', color='gray'),
)
ax.set_xlabel('Threshold τ')
ax.set_ylabel('Expected cost (log10 dollars)')
ax.set_title('Expected dollar cost of the EWS as a function of τ')
ax.legend(loc='best', fontsize=10)
plt.tight_layout()
plt.show()

comparison_rows = []
for label, t in [('Cost-minimising', tau_cost),
                 ('F1-optimal',       tau_f1),
                 ('Capacity (≤5)',   tau_cap)]:
    if np.isnan(t):
        continue
    idx = int(round((t - 0.01) * 100))
    idx = max(0, min(idx, len(tau_grid_cost) - 1))
    comparison_rows.append({
        'τ':       round(float(tau_grid_cost[idx]), 2),
        'Flagged': int((logit_proba_test >= tau_grid_cost[idx]).sum()),
        'FN':      int(fn_counts[idx]),
        'FP':      int(fp_counts[idx]),
        'Cost ($M)': int(round(cost_curve[idx] / 1e6)),
        'Strategy': label,
    })
comparison_tbl = pd.DataFrame(comparison_rows).set_index('Strategy')
print(comparison_tbl)


The cost-minimising τ typically sits below both the F1-optimal τ and the capacity-constrained τ, because the 50-billion-to-2-million asymmetry pushes the optimiser to accept many more false alarms in exchange for catching even one additional crisis. The F1-optimal τ is a middle-ground rule that ignores dollar weights. The capacity-constrained τ is the highest of the three because it is artificially pinned by the five-mission budget rather than by the cost structure.

My recommendation to the Division Chief is the cost-minimising τ, paired with a corresponding expansion of mission capacity. The capacity-constrained rule is the right *default* for the Phase 3 operational memo because it respects a binding budget; but the whole point of Phase 4 is that the five-mission budget itself is a policy lever, not a physical constraint. If the cost-minimising τ flags twelve countries and the capacity-constrained τ flags five, the memo to the Managing Director should propose funding seven additional missions that quarter. The math says the marginal missed crisis is worth many multiples of the marginal mission budget, and a decision-support system worth its name should surface that tradeoff rather than hide it behind a five-country ceiling.